# StART — Enterprise Agentic Model Review (v2.0.0)

The audit-ready operating system. The review runs as explicit layers
(Data → Model → Validation → Governance → AI-Engineering → Evidence →
Reporting), each emitting status, runtime, findings, artifacts, and evidence
IDs. Produces an enterprise dashboard, governance findings, executable
AI-engineering controls, and a review graph — from one execution flow.

Deterministic by default. Public providers (OpenAI/Anthropic/Grok) and the
enterprise gateway are strictly isolated.


## 1. Options


In [ ]:
OPTIONS = {
    'dataset_path': '',
    'target_column': 'attrition',
    'split_strategy': 'stratified',
    'architecture': 'mlp',        # mlp | residual_mlp | wide_deep
    'activation': 'relu',
    'agent_mode': 'deterministic',  # deterministic | llm
    'provider': 'none',           # none | openai | anthropic | grok | enterprise_llm_gateway
    'enterprise_mode': True,
    'governance_mode': True,
    'run_dl': True,
    'cnn_preset': 'simple_cnn_small',  # small | medium | deep | custom
}
print(OPTIONS)

## 2. Optional widgets (enterprise / CNN / governance)


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    w_target = widgets.Text(value=OPTIONS['target_column'], description='target')
    w_arch = widgets.Dropdown(options=['mlp','residual_mlp','wide_deep'], value=OPTIONS['architecture'], description='arch')
    w_act = widgets.Dropdown(options=['relu','leaky_relu','gelu','tanh','selu','elu'], value=OPTIONS['activation'], description='activation')
    w_mode = widgets.Dropdown(options=['deterministic','llm'], value=OPTIONS['agent_mode'], description='agent')
    w_prov = widgets.Dropdown(options=['none','openai','anthropic','grok','enterprise_llm_gateway'], value=OPTIONS['provider'], description='provider')
    w_ent = widgets.Checkbox(value=OPTIONS['enterprise_mode'], description='enterprise')
    w_gov = widgets.Checkbox(value=OPTIONS['governance_mode'], description='governance')
    w_dl = widgets.Checkbox(value=OPTIONS['run_dl'], description='run_dl')
    w_cnn = widgets.Dropdown(options=['simple_cnn_small','simple_cnn_medium','simple_cnn_deep','custom'], value=OPTIONS['cnn_preset'], description='cnn')
    display(w_target, w_arch, w_act, w_mode, w_prov, w_ent, w_gov, w_dl, w_cnn)
    _W = (w_target, w_arch, w_act, w_mode, w_prov, w_ent, w_gov, w_dl, w_cnn)
except ImportError:
    _W = None
    print('ipywidgets not installed; using OPTIONS above.')

## 3. CNN configuration (transparent, evidence-backed)


In [ ]:
from start.modeling.vision_models import config_from_preset, describe_cnn
if _W is not None:
    wt, wa, wac, wm, wp, we, wg, wd, wc = _W
    OPTIONS.update(target_column=wt.value, architecture=wa.value, activation=wac.value,
                   agent_mode=wm.value, provider=wp.value, enterprise_mode=we.value,
                   governance_mode=wg.value, run_dl=wd.value, cnn_preset=wc.value)
preset = OPTIONS['cnn_preset']
if preset == 'custom':
    cfg = config_from_preset('simple_cnn_small', n_blocks=3, base_channels=24, kernel_size=5)
    cnn_descriptor = describe_cnn('simple_cnn', 3, 32, 3, config=cfg)
else:
    cnn_descriptor = describe_cnn(preset, 3, 32, 3)
import pandas as pd
display(pd.DataFrame([cnn_descriptor]).T.rename(columns={0: 'value'}))

## 4. Load data + resolve provider (strict trust-domain separation)


In [ ]:
path = OPTIONS['dataset_path'].strip()
target = OPTIONS['target_column'] or None
if path:
    from start.data.loaders import load_any_tabular
    df = load_any_tabular(path)
else:
    from start.modeling.data import load_attrition_dataset
    df = load_attrition_dataset(seed=42)
    target = target or 'attrition'
llm = None
if OPTIONS['agent_mode'] == 'llm' and OPTIONS['provider'] not in ('none',''):
    from start.core.config import LLMConfig
    from start.providers.llm import get_llm_provider
    from start.providers.trust_domains import trust_domain
    domain = trust_domain(OPTIONS['provider']).value
    expected = domain if domain in ('public','private') else None
    llm = get_llm_provider(LLMConfig(provider=OPTIONS['provider']), expected_domain=expected)
    print('provider', OPTIONS['provider'], '|', domain, 'domain')
else:
    print('Deterministic mode — no key required.')
print(f'{len(df)} rows x {df.shape[1]} columns | target: {target}')

## 5. Run the enterprise layered review


In [ ]:
from start.modeling.enterprise_orchestrator import EnterpriseReviewOrchestrator
def show_layer(lr):
    if lr.status != 'running':
        print(f'  {lr.name:16s} {lr.status} {lr.runtime_seconds:.3f}s '
              f'findings={len(lr.findings)} artifacts={len(lr.artifacts)} evidence={len(lr.evidence_ids)}')
orch = EnterpriseReviewOrchestrator(on_layer=show_layer)
outcome = orch.run(df, user_target=target, split_strategy=OPTIONS['split_strategy'],
                   agent_mode=OPTIONS['agent_mode'], llm=llm, output_root='start_output',
                   run_dl=OPTIONS['run_dl'], enterprise_mode=OPTIONS['enterprise_mode'],
                   cnn_config=cnn_descriptor, seed=42)

## 6. Governance findings, AI-engineering controls, audit package


In [ ]:
import pandas as pd
s = outcome.findings_register.summary()
print('findings:', s)
display(pd.DataFrame(outcome.findings_register.to_list()))
display(pd.DataFrame(outcome.ai_engineering.summary_rows()))
print('dashboard:', outcome.dashboard_paths['html'])
print('review graph:', outcome.graph_paths)
print('evidence critique:', 'PASSED' if outcome.critique_ok else 'FAILED')

## 7. Visible co-pilot (v2.1.1)

LLM activation, agent reasoning traces, AI-engineering control surface, and
the artifact catalog — the same visibility the terminal shows.


In [ ]:
# Section A: LLM activation
if outcome.activation_report is not None:
    print(outcome.activation_report.render_terminal())

In [ ]:
# Section K: agent reasoning traces
import pandas as pd
if outcome.trace_log is not None and outcome.trace_log.traces:
    display(pd.DataFrame(outcome.trace_log.to_list()))

In [ ]:
# Sections L/M: AI-engineering control surface
display(pd.DataFrame(outcome.ai_engineering.control_surface()))

In [ ]:
# Section N: artifact catalog
if outcome.artifact_registry is not None:
    display(pd.DataFrame(outcome.artifact_registry.to_list()))

## Model execution (v2.1.1): split, metrics, training, explainability


In [ ]:
import pandas as pd
ce = outcome.copilot_execution
if ce is not None:
    print('Train/Test/OOS split:')
    display(pd.DataFrame(ce.split_table))
    print('Metrics by split:')
    display(pd.DataFrame(ce.metrics_by_split).T)
    print(f'Generalization gap (train - OOS): {ce.generalization_gap}')
    print(f'Explainability method: {ce.explainability_method}')
    display(pd.DataFrame(ce.global_importance))
else:
    print('Model execution skipped (run_dl=False or non-tabular).')

## Live committee (v2.2.0): roster, adapters, sensitivity, review journey


In [ ]:
import pandas as pd
from start.agent_roster import render_agent_roster, roster_as_list
print(render_agent_roster())
display(pd.DataFrame(roster_as_list()))

In [ ]:
display(pd.DataFrame(outcome.ai_engineering.control_surface()))

In [ ]:
if outcome.sensitivity is not None:
    rows = outcome.sensitivity.to_dict()['rows']
    display(pd.DataFrame(rows)[['feature','shock','baseline','metric','drift','risk_impact']])
else:
    print('No sensitivity run (enable run_dl on a tabular cohort).')

In [ ]:
from start.review_session import Decision, ReviewSession
from start.agent_dialogue import AgentContext, ask_agent
session = ReviewSession(run_id=outcome.run_id)
session.record_decision(Decision(key='architecture', prompt='Model family?',
    recommended='mlp', user_value='mlp', effective='mlp', choice='accept',
    rationale='Small tabular dataset favors a simpler MLP.'))
ctx = AgentContext(agent='ArchitectureReviewAgent', recommendation='mlp',
    reason='Small tabular dataset favors a simpler MLP.',
    risk_if_ignored='Higher overfitting risk.', checkpoint='architecture')
ask_agent('ArchitectureReviewAgent', 'Why not wide_deep?', ctx, session)
display(pd.DataFrame(session.to_dict()['conversations']))

In [ ]:
from start.reporting.review_transcript import write_transcript
paths = write_transcript(session, '/tmp/start_notebook', outcome.run_id, sensitivity=outcome.sensitivity)
print('Transcript written:', paths)